In [1]:
import pandas as pd
data = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
data.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [2]:
data.isna().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [3]:
len(data)

7613

In [4]:
data['keyword'].value_counts()

keyword
fatalities               45
deluge                   42
armageddon               42
damage                   41
body%20bags              41
                         ..
forest%20fire            19
epicentre                12
threat                   11
inundation               10
radiation%20emergency     9
Name: count, Length: 221, dtype: int64

In [5]:
data["keyword"] = data["keyword"].fillna("none")
data = data.drop(columns=["location"])

In [6]:
data.head()

,id,keyword,text,target
0,1,none,Our Deeds are the Reason of this #earthquake M...,1
1,4,none,Forest fire near La Ronge Sask. Canada,1
2,5,none,All residents asked to 'shelter in place' are ...,1
3,6,none,"13,000 people receive #wildfires evacuation or...",1
4,7,none,Just got sent this photo from Ruby #Alaska as ...,1


In [7]:
data = data.drop(columns=["id"])

In [8]:
data['text'].isna().sum()

np.int64(0)

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

In [10]:
data["text_input"] = data["keyword"] + " " + data["text"]
X = data["text_input"]
y = data["target"]

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("clf", LogisticRegression(
        max_iter=200,
        C=1.0,
        solver="liblinear"
    ))
])

In [13]:
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                                 stop_words='english')),
                ('clf', LogisticRegression(max_iter=200, solver='liblinear'))])

In [14]:
val_preds = model.predict(X_val)
print("F1:", f1_score(y_val, val_preds))

F1: 0.7702371218315618


In [15]:
test_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")
test_df["keyword"] = test_df["keyword"].fillna("none")

test_df["text_input"] = test_df["keyword"] + " " + test_df["text"]

test_preds = model.predict(test_df["text_input"])

submission = pd.DataFrame({
    "id": test_df["id"],
    "target": test_preds
})

submission.to_csv("submission.csv", index=False)

In [16]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

In [17]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
import torch

In [18]:
train_df = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/nlp-getting-started/test.csv")

In [19]:
train_df["keyword"] = train_df["keyword"].fillna("")
test_df["keyword"] = test_df["keyword"].fillna("")

In [20]:
train_df["text"] = train_df["keyword"] + " " + train_df["text"]
test_df["text"]  = test_df["keyword"] + " " + test_df["text"]

In [21]:
train_df = train_df[["text", "target"]]
test_df  = test_df[["id", "text"]]

In [22]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset  = Dataset.from_pandas(test_df)

In [23]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [24]:
def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )
    return tokens

In [25]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/7613 [00:00<?, ? examples/s]

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

In [26]:
print(train_dataset.column_names)

['text', 'target', 'input_ids', 'attention_mask']


In [27]:
train_dataset = train_dataset.rename_column("target", "labels")

In [28]:
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask"]
)

In [29]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [30]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
)

In [32]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

In [33]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,1.093611
100,0.850185
150,0.774417
200,0.848647
250,0.777093
300,0.671958
350,0.721786
400,0.695546
450,0.636488
500,0.615229


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=714, training_loss=0.7092115959199536, metrics={'train_runtime': 108.2287, 'train_samples_per_second': 211.025, 'train_steps_per_second': 6.597, 'total_flos': 354485346057336.0, 'train_loss': 0.7092115959199536, 'epoch': 3.0})

In [34]:
print(train_dataset.column_names)

['text', 'labels', 'input_ids', 'attention_mask']


In [35]:
preds = trainer.predict(test_dataset)
logits = preds.predictions
y_pred = np.argmax(logits, axis=1)

In [36]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": y_pred
})

submission.to_csv("submission2.csv", index=False)

# 0.83573 in Final Submissions